# 05 · Previsões com o modelo vigente e teste integrado

**Etapa 8.** Carrega o modelo vigente indicado em `models/modelo_vigente.json` (pipeline com pré-processamento + features do ciclo em que foi promovido), aplica o **mesmo ETL** ao conjunto **reservado** `teste_final.csv` (nov–dez/2012, nunca usado em decisões) e gera as previsões.
Em seguida verifica automaticamente o ciclo (votação refeita, dois algoritmos treinados, decisão registrada) e compara todos os ciclos registrados no MLflow.

In [ ]:
# Setup: garante caminhos consistentes (nbconvert executa com cwd = notebooks/)
import os, sys, warnings
from pathlib import Path

RAIZ = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "config" / "config.yaml").exists())
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
warnings.filterwarnings("ignore")

import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Image
import mlflow

from src import utils
cfg = utils.carregar_config()
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
print("Raiz do projeto:", RAIZ)

In [ ]:
from src import preprocessamento as pp, modelagem, promocao, ingestao

ciclo = utils.carregar_ciclo()
DIR_EV = utils.dir_evidencias(ciclo["cycle_id"])
manifesto = promocao.carregar_manifesto()
assert manifesto is not None, "Não há modelo vigente"
modelo = promocao.carregar_modelo_vigente(manifesto)   # valida o SHA-256 do artefato
print(f"Modelo vigente: {manifesto['modelo_id']} | dados {manifesto['data_version']} | features: {manifesto['features']}")

In [ ]:
teste_bruto = ingestao.carregar_teste_final(cfg)
teste, _ = pp.preparar(teste_bruto, cfg)
assert teste["dteday"].min() > pd.Timestamp("2012-10-31"), "Teste reservado se sobrepõe aos lotes de treino"

previsto = modelagem.prever(modelo, teste)
metricas_teste = modelagem.avaliar(teste["cnt"], previsto)
saida = teste[["dteday", "hr", "cnt"]].assign(previsto=np.round(previsto, 1),
                                               erro_abs=lambda d: (d["cnt"] - d["previsto"]).abs(),
                                               modelo_id=manifesto["modelo_id"], cycle_id=ciclo["cycle_id"])
saida.to_csv(DIR_EV / "previsoes_teste_final.csv", index=False)
print("Avaliação final no teste reservado:", {k: round(v, 3) for k, v in metricas_teste.items()})
display(saida.head(10))

fig, ax = plt.subplots(figsize=(16, 3.6))
janela = saida.head(24 * 14)
ts = janela["dteday"] + pd.to_timedelta(janela["hr"], unit="h")
ax.plot(ts, janela["cnt"], color="black", lw=1, label="real")
ax.plot(ts, janela["previsto"], lw=1, label=f"previsto ({manifesto['modelo_id']})")
ax.set_title("Teste reservado – primeiras 2 semanas de novembro/2012"); ax.legend()
fig.tight_layout(); fig.savefig(DIR_EV / "previsoes_teste_final.png", dpi=110); plt.show()

## Teste integrado do ciclo (verificações automáticas no MLflow)

In [ ]:
exp_id = utils.configurar_mlflow(cfg)
runs = mlflow.search_runs(experiment_ids=[exp_id], output_format="pandas")
do_ciclo = runs[runs["tags.cycle_id"] == ciclo["cycle_id"]]
etapas = do_ciclo["tags.etapa"].value_counts().to_dict()
print("Runs do ciclo:", etapas)
assert etapas.get("02_votacao", 0) >= 1, "Votação não foi refeita neste ciclo"
assert etapas.get("03_treino", 0) >= 2, "Os dois algoritmos não foram treinados neste ciclo"
assert do_ciclo["tags.algoritmo"].dropna().nunique() >= 2
assert etapas.get("04_decisao", 0) >= 1, "Decisão de promoção não registrada"
assert manifesto["ultima_avaliacao"]["cycle_id"] == ciclo["cycle_id"]
print("Teste integrado OK: votação refeita, Ridge e RandomForest treinados, decisão",
      manifesto["ultima_avaliacao"]["decisao"], "registrada.")

with mlflow.start_run(run_name=f"{ciclo['cycle_id']}_05_previsoes", tags=utils.tags_ciclo(ciclo, "05_previsoes")):
    mlflow.log_params({"modelo_vigente": manifesto["modelo_id"], "conjunto": cfg["dados"]["teste_final"],
                       "n_previsoes": len(saida)})
    mlflow.log_metrics({f"teste_{k}": v for k, v in metricas_teste.items()})
    mlflow.log_artifact(str(DIR_EV / "previsoes_teste_final.csv"), "previsoes")
    mlflow.log_artifact(str(DIR_EV / "previsoes_teste_final.png"), "previsoes")

## Comparação entre ciclos (histórico recuperado do MLflow)

In [ ]:
runs = mlflow.search_runs(experiment_ids=[exp_id], output_format="pandas")
linhas = []
for cid, g in runs.groupby("tags.cycle_id"):
    vot = g[g["tags.etapa"] == "02_votacao"].sort_values("start_time").tail(1)
    tre = g[g["tags.etapa"] == "03_treino"]
    dec = g[g["tags.etapa"] == "04_decisao"].sort_values("start_time").tail(1)
    prev = g[g["tags.etapa"] == "05_previsoes"].sort_values("start_time").tail(1)
    etl = g[g["tags.etapa"] == "01_etl"].sort_values("start_time").tail(1)
    linha = {"cycle_id": cid, "data_version": g["tags.data_version"].iloc[0],
             "gatilho": g["tags.gatilho"].iloc[0],
             "linhas": etl["metrics.linhas_consolidadas"].iloc[0] if len(etl) else None,
             "n_features_eleitas": vot["metrics.n_features_eleitas"].iloc[0] if len(vot) else None,
             "features_eleitas": vot["params.features_eleitas"].iloc[0] if len(vot) else None}
    for _, r in tre.iterrows():
        linha[f"val_mae_{r['tags.algoritmo']}"] = r["metrics.val_mae"]
    if len(dec):
        linha.update({"vencedor_ciclo": dec["params.vencedor_ciclo"].iloc[0], "decisao": dec["tags.decisao"].iloc[0],
                      "vigente_val_mae": dec.get("metrics.vigente_val_mae", pd.Series([None])).iloc[0],
                      "melhoria": dec.get("metrics.melhoria", pd.Series([None])).iloc[0]})
    if len(prev):
        linha["teste_mae_vigente"] = prev["metrics.teste_mae"].iloc[0]
        linha["modelo_vigente"] = prev["params.modelo_vigente"].iloc[0]
    if len(etl) and "metrics.drift_psi_maximo" in etl:
        linha["drift_psi_maximo"] = etl["metrics.drift_psi_maximo"].iloc[0]
    linhas.append(linha)

historico = pd.DataFrame(linhas)
historico["_n"] = historico["cycle_id"].str.extract(r"(\d+)").astype(int)
historico = historico.sort_values("_n").drop(columns="_n").reset_index(drop=True)
historico.to_csv(utils.DIR_EVIDENCIAS / "historico_ciclos.csv", index=False)
display(historico.drop(columns=["features_eleitas"]).round(3))

In [ ]:
# O que mudou em relação ao ciclo anterior?
n = ciclo["numero_execucao"]
if n > 1:
    ant = utils.ler_json(utils.DIR_EVIDENCIAS / f"ciclo_{n-1}" / "features_eleitas.json")
    atu = utils.ler_json(DIR_EV / "features_eleitas.json")
    mudancas = {
        "ciclo_anterior": f"ciclo_{n-1}", "ciclo_atual": ciclo["cycle_id"],
        "data_version": [ant["data_version"], atu["data_version"]],
        "linhas_treino": [ant["particao"]["treino"][2], atu["particao"]["treino"][2]],
        "features_entraram": sorted(set(atu["features_eleitas"]) - set(ant["features_eleitas"])),
        "features_sairam": sorted(set(ant["features_eleitas"]) - set(atu["features_eleitas"])),
        "decisao": manifesto["ultima_avaliacao"]["decisao"],
        "modelo_vigente": manifesto["modelo_id"],
    }
    utils.salvar_json(mudancas, DIR_EV / "mudancas_vs_ciclo_anterior.json")
    print(json.dumps(mudancas, indent=2, ensure_ascii=False))
else:
    print("Primeiro ciclo: não há ciclo anterior para comparar.")

In [ ]:
# Resumo do ciclo (exibido no resumo da execução do GitHub Actions)
dec = utils.ler_json(DIR_EV / "decisao_promocao.json")
drift = utils.ler_json(DIR_EV / "drift_relatorio.json")
comp = pd.read_csv(DIR_EV / "competicao_modelos.csv")
eleitas_ciclo = utils.ler_json(DIR_EV / "features_eleitas.json")["features_eleitas"]
md = [f"## Ciclo `{ciclo['cycle_id']}` · dados `{ciclo['data_version']}` · gatilho `{ciclo['gatilho']}`", "",
      f"- Lotes: {', '.join(ciclo['lotes_ciclo'])} (novo: {ciclo['lote_novo'] or 'nenhum'})",
      f"- Features eleitas ({len(eleitas_ciclo)}): `{', '.join(eleitas_ciclo)}`", "",
      "| Algoritmo | MAE val | RMSE val | R² val |", "|---|---|---|---|"]
md += [f"| {r.algoritmo} | {r.val_mae:.3f} | {r.val_rmse:.3f} | {r.val_r2:.3f} |" for r in comp.itertuples()]
md += ["", f"**Decisão: {dec['decisao']}** — {dec['motivo']}", "",
       f"- Modelo vigente: `{manifesto['modelo_id']}` · MAE no teste reservado: **{metricas_teste['mae']:.3f}**"]
if drift:
    md.append(f"- Drift: {'⚠️ ALERTA em ' + ', '.join(drift['variaveis_com_drift']) if drift['alerta'] else 'sem alerta'}"
              f" (PSI máx. {drift['psi_maximo']:.3f})")
(DIR_EV / "resumo_ciclo.md").write_text("\n".join(md) + "\n", encoding="utf-8")
print("\n".join(md))